# Модуль 1: обучение извлечению причин и следствий
Выберите **Среда выполнения → Сменить среду выполнения → GPU**. Загрузите пакет обучения с готовыми train/val/test. Источник данных не требуется. Test используется только после выбора весов по validation. F1 на синтетике не измеряет качество на реальных документах.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет; выберите GPU")
assert torch.cuda.is_available(), "Включите GPU перед обучением"

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io, os, uuid
uploaded = files.upload()
assert len(uploaded) == 1, "Загрузите один module1-training.zip"
work = Path("/content") / ("module1-" + uuid.uuid4().hex[:8])
work.mkdir()
with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
    assert sum(i.file_size for i in z.infolist()) < 100_000_000
    for item in z.infolist():
        target = (work / item.filename).resolve()
        assert target.is_relative_to(work.resolve()), "Недопустимый путь в ZIP"
    z.extractall(work)
os.chdir(work)
print(work)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)

In [ ]:
import json
print("Dataset:", [p.name for p in (work / "dataset").iterdir()])
subprocess.run([sys.executable, "-m", "src.causal_extraction.train_module", "--dataset", "dataset", "--output", "run"], check=True)

In [ ]:
metrics = json.loads((work / "run/metrics.json").read_text())
print(json.dumps(metrics, indent=2))
import shutil
archive = shutil.make_archive(str(work / "module1-result"), "zip", work / "run")
files.download(archive)

Распакуйте результат в `models/runs/politics-2000/` проекта. Выполните `python -m src.causal_extraction.activate models/runs/politics-2000` и перезапустите веб-сервер. Исходные веса не заменяются. Повторный подбор параметров по test делает его валидационной выборкой; для окончательной оценки нужен новый независимый тест.